In [1]:
#Import Libraries

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score

import torch
import torch.nn as nn
import torch.optim as optim

import joblib


In [3]:
#Load Dataset

df = pd.read_csv(r"C:\Users\91787\Desktop\Churn_chatbot\Telco-Customer-Churn.csv")
df.head()

,gender,SeniorCitizen,Unnamed: 2,Unnamed: 3,tenure,Unnamed: 5,Unnamed: 6,InternetService,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Contract,Unnamed: 15,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,NaN,NaN,1,NaN,NaN,DSL,NaN,NaN,NaN,NaN,NaN,NaN,Month-to-month,NaN,Electronic check,29.85,29.85,No
1,Male,0,NaN,NaN,34,NaN,NaN,DSL,NaN,NaN,NaN,NaN,NaN,NaN,One year,NaN,Mailed check,56.95,1889.5,No
2,Male,0,NaN,NaN,2,NaN,NaN,DSL,NaN,NaN,NaN,NaN,NaN,NaN,Month-to-month,NaN,Mailed check,53.85,108.15,Yes
3,Male,0,NaN,NaN,45,NaN,NaN,DSL,NaN,NaN,NaN,NaN,NaN,NaN,One year,NaN,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,NaN,NaN,2,NaN,NaN,Fiber optic,NaN,NaN,NaN,NaN,NaN,NaN,Month-to-month,NaN,Electronic check,70.70,151.65,Yes


In [5]:
#Remove Unnamed Columns

df = df.loc[:, ~df.columns.str.contains("Unnamed")]
df.head()


,gender,SeniorCitizen,tenure,InternetService,Contract,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,1,DSL,Month-to-month,Electronic check,29.85,29.85,No
1,Male,0,34,DSL,One year,Mailed check,56.95,1889.5,No
2,Male,0,2,DSL,Month-to-month,Mailed check,53.85,108.15,Yes
3,Male,0,45,DSL,One year,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,2,Fiber optic,Month-to-month,Electronic check,70.70,151.65,Yes


In [7]:
#Handle Data Types

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df.dropna(inplace=True)


In [9]:
#Encode Categorical Columns

label_encoders = {}

categorical_cols = ["gender", "InternetService", "Contract", "PaymentMethod", "Churn"]

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le


In [11]:
#Split Features & Target

X = df.drop("Churn", axis=1)
y = df["Churn"]


In [13]:
#Train–Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [15]:
#Feature Scaling

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

joblib.dump(scaler, "scaler.pkl")


['scaler.pkl']

In [17]:
#Convert to PyTorch Tensors

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1,1)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1,1)


In [19]:
#Define ANN Model

class ChurnANN(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(8, 32)
        self.bn1 = nn.BatchNorm1d(32)
        self.fc2 = nn.Linear(32, 16)
        self.bn2 = nn.BatchNorm1d(16)
        self.fc3 = nn.Linear(16, 1)

    def forward(self, x):
        x = torch.relu(self.bn1(self.fc1(x)))
        x = torch.relu(self.bn2(self.fc2(x)))
        return self.fc3(x)


In [21]:
#Initialize Model, Loss & Optimizer

model = ChurnANN()
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [31]:
#Train the Model

epochs = 140

for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    
    loss.backward()
    optimizer.step()
    
    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")


Epoch 10/140, Loss: 0.4404
Epoch 20/140, Loss: 0.4365
Epoch 30/140, Loss: 0.4331
Epoch 40/140, Loss: 0.4299
Epoch 50/140, Loss: 0.4271
Epoch 60/140, Loss: 0.4247
Epoch 70/140, Loss: 0.4224
Epoch 80/140, Loss: 0.4204
Epoch 90/140, Loss: 0.4185
Epoch 100/140, Loss: 0.4169
Epoch 110/140, Loss: 0.4153
Epoch 120/140, Loss: 0.4139
Epoch 130/140, Loss: 0.4125
Epoch 140/140, Loss: 0.4112


In [35]:
#Model Evaluation

model.eval()

with torch.no_grad():
    test_outputs = model(X_test_tensor)
    preds = torch.sigmoid(test_outputs)
    preds = (preds >= 0.5).int()

accuracy = accuracy_score(y_test_tensor, preds)
print("Test Accuracy:", accuracy)


Test Accuracy: 0.7874911158493249


In [41]:
#Save Trained Model
torch.save(model.state_dict(), "churn_model.pth")
